# Training a Classifier using 4 Different Algorithms

**Dataset:** Breast Cancer Wisconsin (already built into scikit-learn, so nothing to download)

**Goal:** Predict whether a tumour is *malignant* (cancer) or *benign* (not cancer).

**Algorithms compared:**
1. Logistic Regression
2. K-Nearest Neighbours (KNN)
3. Decision Tree
4. Random Forest

**Comparison metrics:** Precision, Recall, F1-score

## Step 1: Import the libraries

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier

from sklearn.metrics import precision_score, recall_score, f1_score, accuracy_score
from sklearn.metrics import confusion_matrix, classification_report

## Step 2: Load the dataset

`X` = the input features (30 measurements of the tumour).
`y` = the answer we want to predict (0 = malignant, 1 = benign).

In [ ]:
data = load_breast_cancer()

X = pd.DataFrame(data.data, columns=data.feature_names)
y = pd.Series(data.target)

print("Number of rows (patients):", X.shape[0])
print("Number of columns (features):", X.shape[1])
print("Classes:", data.target_names)

X.head()

### How many of each class do we have?

It is always good to check this. If one class is much bigger than the other,
accuracy alone can be misleading.

In [ ]:
print(y.value_counts())
print()
print("0 = malignant, 1 = benign")

## Step 3: Split the data into 80% train and 20% test

- `test_size=0.2` --> 20% goes to testing, 80% to training
- `random_state=42` --> gives the same split every time we run it
- `stratify=y` --> keeps the same class ratio in train and test

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print("Training rows:", X_train.shape[0])
print("Testing rows :", X_test.shape[0])

## Step 4: Scale the features

The columns have very different ranges (some are 0.001, some are 2000).
Logistic Regression and KNN work much better when all columns are on a similar scale.

**Important:** we `fit` the scaler only on the training data, then `transform` both.
If we fit on the whole dataset, information from the test set leaks into training.

In [ ]:
scaler = StandardScaler()

X_train_scaled = scaler.fit_transform(X_train)   # learn + apply on train
X_test_scaled  = scaler.transform(X_test)        # only apply on test

## Step 5: Train all 4 models and measure the results

We put the 4 models in a dictionary and train them one by one in a loop,
so we do not have to repeat the same code 4 times.

In [ ]:
models = {
    "Logistic Regression": LogisticRegression(max_iter=5000),
    "KNN":                 KNeighborsClassifier(n_neighbors=5),
    "Decision Tree":       DecisionTreeClassifier(random_state=42),
    "Random Forest":       RandomForestClassifier(random_state=42)
}

results = []
predictions = {}

for name, model in models.items():
    model.fit(X_train_scaled, y_train)              # train
    y_pred = model.predict(X_test_scaled)           # predict on unseen test data
    predictions[name] = y_pred

    results.append({
        "Model":     name,
        "Accuracy":  accuracy_score(y_test, y_pred),
        "Precision": precision_score(y_test, y_pred),
        "Recall":    recall_score(y_test, y_pred),
        "F1 Score":  f1_score(y_test, y_pred)
    })

    print(name, "-> done")

## Step 6: Compare the results

- **Precision** = out of all the patients we *said* were benign, how many really were.
- **Recall** = out of all the patients who really were benign, how many we caught.
- **F1 Score** = a single balanced score combining precision and recall.

In [ ]:
results_df = pd.DataFrame(results).sort_values("F1 Score", ascending=False)
results_df = results_df.round(4).reset_index(drop=True)

results_df

## Step 7: Plot the comparison

In [ ]:
results_df.plot(
    x="Model",
    y=["Precision", "Recall", "F1 Score"],
    kind="bar",
    figsize=(9, 5)
)

plt.title("Comparison of the 4 Classifiers")
plt.ylabel("Score")
plt.ylim(0.8, 1.0)        # zoomed in, because all scores are high
plt.xticks(rotation=15)
plt.grid(axis="y", alpha=0.3)
plt.legend(loc="lower right")
plt.tight_layout()
plt.show()

## Step 8: Which model is the best?

We pick the best model using the **F1 score**, because it balances
precision and recall instead of looking at only one of them.

In [ ]:
best_row  = results_df.iloc[0]
best_name = best_row["Model"]

print("Best model :", best_name)
print("Precision  :", best_row["Precision"])
print("Recall     :", best_row["Recall"])
print("F1 Score   :", best_row["F1 Score"])

## Step 9: Look at the best model in detail

The **confusion matrix** shows exactly what the model got right and wrong:

```
                 Predicted malignant   Predicted benign
Actual malignant        correct             mistake
Actual benign           mistake             correct
```

In [ ]:
y_pred_best = predictions[best_name]

print("Confusion matrix for:", best_name)
print(confusion_matrix(y_test, y_pred_best))
print()
print(classification_report(y_test, y_pred_best, target_names=data.target_names))

## Conclusion

Write your observation here after running the notebook, for example:

- The best model was **<fill in the name printed in Step 8>** with the highest F1 score.
- Logistic Regression and Random Forest usually do very well on this dataset.
- The Decision Tree is usually the weakest, because a single tree overfits the
  training data, while Random Forest averages many trees and is more stable.
- Recall matters a lot in medical problems: missing a real cancer case (low recall
  on the malignant class) is far more dangerous than a false alarm.